# Generate TikZ diagrams from pipeline data

This notebook consumes shell/status data and emits TikZ fragments for the pipeline diagrams. It stays read-only with respect to the running download.

In [ ]:
import json, pathlib

DIAGRAM_DIR = pathlib.Path(r'C:\R\LeafOS0.2.1\ProjectLeaf\design\diagrams')
print('Diagram dir exists:', DIAGRAM_DIR.exists())


## 1. Parse or create a sample status log

Replace `STATUS_PATH` with real JSON-lines output from the downloader.

In [ ]:
STATUS_PATH = DIAGRAM_DIR / 'sample-status.jsonl'
sample = [
    {'file': 'Qwen3.6-27B-A.gguf', 'size_gb': 24.0, 'done_gb': 0.5, 'speed_mbs': 5.1},
    {'file': 'Qwen3.6-27B-B.gguf', 'size_gb': 29.8, 'done_gb': 0.3, 'speed_mbs': 3.2},
    {'file': 'MiniCPM5-1B.gguf',   'size_gb': 0.7,  'done_gb': 0.7, 'speed_mbs': 8.8},
]
with open(STATUS_PATH, 'w') as f:
    for row in sample:
        f.write(json.dumps(row) + '\n')
print('Wrote', STATUS_PATH)


In [ ]:
def load_status(path: pathlib.Path):
    if not path.exists():
        return []
    with open(path) as f:
        return [json.loads(line) for line in f if line.strip()]

rows = load_status(STATUS_PATH)
total_gb = sum(r['size_gb'] for r in rows)
done_gb = sum(r['done_gb'] for r in rows)
agg_mbs = sum(r['speed_mbs'] for r in rows)
print(f'{len(rows)} artifacts | total {total_gb:.2f} GiB | done {done_gb:.2f} GiB | aggregate {agg_mbs:.2f} MiB/s')


## 2. Generate a TikZ bar chart of per-file progress

Copy the printed fragment into a LaTeX document or standalone figure.

In [ ]:
def tikz_progress_bars(rows):
    lines = ['\\begin{tikzpicture}[font=\\sffamily\\small,>=Stealth]']
    y = 0.0
    width_factor = 8.0
    max_size = max(r['size_gb'] for r in rows)
    lines.append('\\draw[->] (0,0.5) -- (10,0.5) node[right]{GiB};')
    for r in rows:
        pct = r['done_gb'] / r['size_gb'] if r['size_gb'] else 0
        bar_len = r['size_gb'] / max_size * width_factor
        fill_len = bar_len * pct
        label = pathlib.Path(r['file']).stem[:25]
        lines.append(r'\\\\node[anchor=east] at (-0.2,-%f) {%s};' % (y, label))
        lines.append(r'\\\\draw[thick] (0,-%f) rectangle (%f,-%f);' % (y, bar_len, y+0.35))
        lines.append(r'\\\\fill[blue!40] (0,-%f) rectangle (%f,-%f);' % (y, fill_len, y+0.35))
        lines.append(r'\\\\node[anchor=west] at (%f+0.1,-%f) {%.1f/%.1f GiB};' % (bar_len, y+0.175, r['done_gb'], r['size_gb']))
        y += 0.7
    lines.append('\\end{tikzpicture}')
    return '\n'.join(lines)

print(tikz_progress_bars(rows))


## 3. Generate a TikZ token-resolution chain from a priority list

Active sources are highlighted; inactive ones are grayed out.

In [ ]:
PRIORITY = [
    ('CLI \texttt{--hf-token}', True),
    ('Env HF\_TOKEN', False),
    ('DPAPI vault', True),
    ('Cached token', False),
    ('Anonymous', True),
]

def tikz_token_resolution(items):
    lines = ['\\begin{tikzpicture}[font=\\sffamily\\small,>=Stealth]']
    for i, (name, active) in items:
        color = 'blue!60' if active else 'gray!60'
        fill = 'blue!5' if active else 'gray!10'
        y = i * 1.2
        lines.append(r'\\\\node[rectangle, rounded corners, draw=%s, fill=%s, thick, minimum width=4cm, minimum height=0.8cm] at (0,-%f) {%s};' % (color, y, name))
        if i > 0:
            lines.append(r'\\\\draw[->, thick] (0,-%f) -- (0,-%f);' % ((i-1)*1.2+0.4, y-0.4))
    lines.append('\\end{tikzpicture}')
    return '\n'.join(lines)

print(tikz_token_resolution(list(enumerate(PRIORITY))))


## 4. (Optional) Render the progress bars with matplotlib

Use this if a PNG preview is needed without compiling LaTeX.

In [ ]:
try:
    import matplotlib.pyplot as plt
except ImportError:
    print('matplotlib not installed; skipping preview')
else:
    labels = [r['file'][:25] for r in rows]
    sizes = [r['size_gb'] for r in rows]
    dones = [r['done_gb'] for r in rows]
    fig, ax = plt.subplots(figsize=(10, 4))
    y = range(len(labels))
    ax.barh(y, sizes, color='lightblue', edgecolor='navy', label='Total')
    ax.barh(y, dones, color='steelblue', label='Done')
    ax.set_yticks(list(y))
    ax.set_yticklabels(labels)
    ax.set_xlabel('GiB')
    ax.set_title('Pack 1 per-file progress')
    ax.legend()
    plt.tight_layout()
    plt.show()


## Notes

- Point the parser at real JSON-lines status records when available.
- The generated TikZ uses only the `tikz` package and standard shapes.
- To drive the notebook from shell output, wrap `download-pack-hftransfer.py` so it prints one JSON object per file update.